# Probabilistic Explanations for Text Classification with `PyProbXAI`

This tutorial demonstrates how to compute and visualize **probable explanations** for text classifiers using `PyProbXAI`.

---

### Why Probable Explanations for NLP?

Standard feature attribution approaches (such as LIME, SHAP, or raw attention maps) score individual words, but leave two critical questions unanswered:
1. **Sufficiency**: If we fix these top words, will the model's prediction remain stable across arbitrary other words?
2. **Absence of Counter-evidence**: Can a model's prediction be overturned simply by adding counter-evidence?

`PyProbXAI` solves this by computing a **minimal set of conditions** (size $\le k$) with an $(\eta, \delta)$-PAC statistical guarantee:
$$\text{Precision}(S) = 1 - \text{Error}(S) \ge 1 - \epsilon$$

The resulting explanation specifies:
* **Active Evidence**: Words that are present and required.
* **Absence Guards**: Words that are absent and forbidden from appearing.


In [ ]:
import os
import sys
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.neural_network import MLPClassifier

# Ensure PyProbXAI is on path
sys.path.insert(0, os.path.abspath(".."))
import PyProbXAI as pe
from PyProbXAI import BlackBoxExplainer


## 1. Dataset & Vocabulary Extraction

We load documents from the 20 Newsgroups benchmark comparing two domains:
* `sci.med` (Medicine)
* `sci.space` (Astronomy & Spaceflight)

We extract the top 50 most discriminant vocabulary terms via $\chi^2$ feature selection.


In [ ]:
CATEGORY_A, CATEGORY_B = "sci.med", "sci.space"
N_WORDS = 50

# Remove headers, footers, and quotes to prevent shortcut learning
remove = ("headers", "footers", "quotes")
train = fetch_20newsgroups(subset="train", categories=[CATEGORY_A, CATEGORY_B], remove=remove)
test = fetch_20newsgroups(subset="test", categories=[CATEGORY_A, CATEGORY_B], remove=remove)

names = np.array(train.target_names)
y_train, y_test = names[train.target], names[test.target]

vectorizer = CountVectorizer(binary=True, stop_words="english", min_df=5)
counts = vectorizer.fit_transform(train.data)
selector = SelectKBest(chi2, k=N_WORDS).fit(counts, y_train)
words = list(vectorizer.get_feature_names_out()[selector.get_support()])

def to_features(docs):
    return selector.transform(vectorizer.transform(docs)).toarray().astype(float)

X_train, X_test = to_features(train.data), to_features(test.data)

print(f"Loaded {len(X_train)} train and {len(X_test)} test documents.")
print(f"Top vocabulary words (50 total):\n{', '.join(words[:15])}...")


## 2. Model Training

We fit a non-linear black-box neural network (`MLPClassifier`) on these 50 features.


In [ ]:
nn = MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=0)
nn.fit(X_train, y_train)
print(f"Neural Network Test Accuracy: {nn.score(X_test, y_test):.2%}")


## 3. Initializing the `BlackBoxExplainer`

We initialize `BlackBoxExplainer`:
* `X_background=X_train`: Used to draw Monte-Carlo perturbations of free words.
* `categorical_features`: All 50 word features.
* `cat_decoders`: Maps numeric values $0 \to \text{'absent'}$ and $1 \to \text{'present'}$.
* `eta=0.15, delta=0.1`: PAC statistical parameters.


In [ ]:
explainer = BlackBoxExplainer(
    nn,
    X_background=X_train,
    categorical_features=list(range(N_WORDS)),
    feature_names=words,
    cat_decoders={j: ["absent", "present"] for j in range(N_WORDS)},
    eta=0.15,
    delta=0.1,
    random_state=0
)


## 4. Explaining a Medical Document (`sci.med`)

Let us select document #164: an inquiry about treating a centipede bite.


In [ ]:
doc_idx = 164
doc_text = test.data[doc_idx].strip()
x_doc = X_test[doc_idx]

# Compute explanation of size at most K=5
explanation = explainer.explain(x_doc, k=5)

# Textual summary of the explaination
print(explanation)

# Visual display
explanation

### Native Interactive Visualization (`e.show_text()`)

In a Jupyter notebook, simply invoking `explanation.show_text(...)` renders an interactive document view:
* **Green highlighted spans**: In-text evidence confirmed by the explainer.
* **Red Absence Guards**: Critical counter-evidence terms forbidden from appearing.
* **Certified Precision Bar**: High-probability bound guarantee ($1 - \epsilon$).


In [ ]:
explanation.show_text(
    doc_text,
    title=f"Explanation for Document #{doc_idx}",
    true_class=y_test[doc_idx],
)


## 5. Explaining a Spaceflight Document (`sci.space`)

Now, let us examine document #2: a discussion regarding planetary propulsion, orbits, and NASA missions.


In [ ]:
space_idx = 2
space_text = test.data[space_idx].strip()
x_space = X_test[space_idx]

space_explanation = explainer.explain(x_space, k=5)
space_explanation.show_text(
    space_text,
    title=f"Explanation for Document #{space_idx}",
    true_class=y_test[space_idx],
)


## 6. Exporting Explanations for Scientific Papers & Reports

Every `TextExplanationView` provides a `.save()` method to export a standalone, self-contained HTML report.


In [ ]:
os.makedirs("../results", exist_ok=True)
view = explanation.show_text(
    doc_text,
    title="Centipede Bite Inquiry Explanation",
    true_class=y_test[doc_idx],
)
saved_path = view.save("../results/text_explanation_demo.html")
print(f"Self-contained HTML report saved to: {saved_path}")


## Summary

`PyProbXAI` makes explainability intuitive and scientifically rigorous:
1. **Guaranteed Precision**: PAC bounds ensure explanations are statistically sufficient.
2. **Positive & Negative Guards**: Identifies both active words and forbidden counter-evidence.
3. **Ergonomic Integration**: One line of Python (`explanation.show_text(text)`) renders interactive displays directly inside Jupyter notebooks.
